In [ ]:
!unzip -q -o ./data/you-tube-comments-signal-vsosai.zip -d ./data/

## Dependencies

In [20]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn

import nltk
from nltk.corpus import stopwords


In [21]:
df_train = pd.read_csv('./data/train.csv')
df_test = pd.read_csv('./data/test.csv')
df_train.head()

,id,comment_text,target
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1
1,167,It's a myth that Neanderthals were stupid. The...,1
2,169,Emma is super white.\nIn other news...,1
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1


## EDA

In [3]:
df_train.shape

(51153, 3)

In [4]:
df_train['target'].value_counts()

target
0    38530
1     7808
2     3125
3     1690
Name: count, dtype: int64

In [5]:
df_train['comment_text'][:10]

0    Eh c kii ce favoritisme lui il a tou led iphon...
1    It's a myth that Neanderthals were stupid. The...
2               Emma is super white.\nIn other news...
3    Chelsea and Aubrey are my favourite on Teen Mo...
4    that mustard colour tho 😻😻 p.s absolutely love...
5    OMG u tagged Dina, never see anyone mention he...
6    when you said you're girl twins would be calle...
7                                 Loved this lousie♥♥♥
8    I'm a man united fan but anyone who can't see ...
9    When Christensen speaks better English than Ro...
Name: comment_text, dtype: object

In [6]:
df_train.isna().sum()

id              0
comment_text    9
target          0
dtype: int64

In [7]:
df_train = df_train.fillna("NA")

## Solutions

In [8]:
X = df_train.drop(columns=['id', 'target'])
y = df_train['target']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
X_train

,comment_text
29554,Fuck life.
44479,woooow
28131,This so so funny
41042,H
34839,Hi
...,...
8444,Clark Gable and George Clooney look the same
31718,The chef is french ?
9697,Pink's reaction would totally be my reaction. ...
37972,ONE HOUR LATER


### CatBoost

In [10]:
model = CatBoostClassifier(
    iterations=50000,
    learning_rate=0.01,
    eval_metric='MultiClass',
    random_seed=42,
    early_stopping_rounds=1000,
    use_best_model=True,
    thread_count=-1,
    task_type='GPU',
    auto_class_weights='Balanced',
    text_features=['comment_text']
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    verbose=1000
)

0:	learn: 1.3757865	test: 1.3736073	best: 1.3736073 (0)	total: 7.51ms	remaining: 6m 15s
1000:	learn: 0.6950581	test: 0.6084727	best: 0.6084727 (1000)	total: 3.79s	remaining: 3m 5s
2000:	learn: 0.6369302	test: 0.5734124	best: 0.5734124 (2000)	total: 7.33s	remaining: 2m 55s
3000:	learn: 0.5974888	test: 0.5535477	best: 0.5535477 (3000)	total: 10.8s	remaining: 2m 49s
4000:	learn: 0.5673470	test: 0.5396326	best: 0.5396326 (4000)	total: 14.3s	remaining: 2m 44s
5000:	learn: 0.5421484	test: 0.5283113	best: 0.5283113 (5000)	total: 17.8s	remaining: 2m 40s
6000:	learn: 0.5204663	test: 0.5197035	best: 0.5197035 (6000)	total: 21.4s	remaining: 2m 36s
7000:	learn: 0.5015961	test: 0.5130328	best: 0.5130328 (7000)	total: 24.8s	remaining: 2m 32s
8000:	learn: 0.4850228	test: 0.5074731	best: 0.5074731 (8000)	total: 28.3s	remaining: 2m 28s
9000:	learn: 0.4698110	test: 0.5018543	best: 0.5018543 (9000)	total: 31.9s	remaining: 2m 25s
10000:	learn: 0.4559653	test: 0.4973895	best: 0.4973894 (9999)	total: 35.3s	

CatBoostClassifier(auto_class_weights='Balanced', early_stopping_rounds=1000, eval_metric='MultiClass', iterations=50000, learning_rate=0.01, random_seed=42, task_type='GPU', text_features=['comment_text'], use_best_model=True)

In [22]:
lm = pd.read_csv('./data/label_mapping.csv')
lm = lm['target_name'].to_dict()
lm

{0: 'ignored', 1: 'applause', 2: 'debate', 3: 'bait'}

In [23]:
df_test.head()

,id,comment_text
0,200,Me with Arabic. I can never have a conversatio...
1,607,if I could subscribe to Dude Perfect a million...
2,1100,We can fortunately say that we lived in the er...
3,1137,"Not an easy Classic to cover, you and your ban..."
4,1139,"I really really really love this cover, hope H..."


In [24]:
df_test['comment_text'].isna().sum()

np.int64(3)

In [25]:
df_test['comment_text'] = df_test['comment_text'].fillna('NA')

In [26]:
subm = df_test.copy()
subm['pred'] = model.predict(df_test[['comment_text']])
subm.head()

,id,comment_text,pred
0,200,Me with Arabic. I can never have a conversatio...,2
1,607,if I could subscribe to Dude Perfect a million...,1
2,1100,We can fortunately say that we lived in the er...,1
3,1137,"Not an easy Classic to cover, you and your ban...",1
4,1139,"I really really really love this cover, hope H...",1


In [28]:
subm['target'] = subm['pred']
subm[['id', 'target']].to_csv("subm.csv", index=False)
subm.head()

,id,comment_text,pred,target
0,200,Me with Arabic. I can never have a conversatio...,2,2
1,607,if I could subscribe to Dude Perfect a million...,1,1
2,1100,We can fortunately say that we lived in the er...,1,1
3,1137,"Not an easy Classic to cover, you and your ban...",1,1
4,1139,"I really really really love this cover, hope H...",1,1


### Linear Models

#### Data preparing

In [29]:
df_train.head()

,id,comment_text,target
0,59,Eh c kii ce favoritisme lui il a tou led iphon...,1
1,167,It's a myth that Neanderthals were stupid. The...,1
2,169,Emma is super white.\nIn other news...,1
3,372,Chelsea and Aubrey are my favourite on Teen Mo...,1
4,373,that mustard colour tho 😻😻 p.s absolutely love...,1
